# 04 · Architecture comparison: BERT-base and RoBERTa-base (E2)

**Runtime:** L4 GPU (bf16). **Run after:** 01 (and ideally 03, so robustness sets are scored too). **Time:** ~1–1.5 h on L4.

Identical protocol to the DeBERTa E2 runs in notebook 05: the same fixed 50K-example training subset, 3 epochs, learning rate 2e-5, effective batch 32, max length 128, and seeds 42/43/44.

In [ ]:
# Setup: Drive, code, packages, paths  (expected runtime: L4)
from google.colab import drive
drive.mount('/content/drive')
import os, sys
BASE = '/content/drive/MyDrive/neural-evidence'
CODE = '/content/drive/MyDrive/Colab Notebooks/neural-evidence'   # your uploaded code folder
REPO_URL = ''
if not os.path.exists(f'{CODE}/ne') and REPO_URL:
    os.system(f'git clone -q {REPO_URL} /content/code'); CODE = '/content/code'
assert os.path.exists(f'{CODE}/ne'), f'Code not found at {CODE}'
os.chdir(CODE); sys.path.insert(0, CODE)
!pip -q install -r requirements.txt sentencepiece protobuf
PROC, RESULTS, CACHE = f'{BASE}/data/processed', f'{BASE}/results', f'{BASE}/cache'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
from ne import runtime
from ne.data import read_jsonl, write_jsonl, eval_sets
runtime.setup()
if runtime.gpu_kind() not in 'L4'.split('/') and 'L4' != 'CPU':
    print('WARNING: expected L4; this runtime is', runtime.gpu_kind(), '- it will still work, just at a different speed/cost')
EVALS = eval_sets(PROC); print('eval sets:', list(EVALS))

In [ ]:
from ne.train import RunConfig, run
E2 = dict(train_file=f'{PROC}/fever/train.jsonl', val_file=f'{PROC}/fever/val.jsonl',
          eval_files={k: v for k, v in EVALS.items() if k != 'fever_val'},
          train_subset=50_000, epochs=3, lr=2e-5, max_length=128, results_dir=RESULTS, scratch_dir='/content/ckpt')
for model_name, name in [('bert-base-uncased', 'bert_base'), ('roberta-base', 'roberta_base')]:
    for seed in (42, 43, 44):
        run(RunConfig(exp='E2', name=name, model_name=model_name, seed=seed, **E2))